# HR-FRGS Ablation Study Notebook

This notebook runs ablation variants of the HR-FRGS pipeline:

1. **Without entropy pruning** in Phase 2 (PPIEntropyFilter).
2. **Without fuzzy-rough set based selection** in Phase 6 (HypergraphFeatureSelector).

It assumes the original pipeline modules and configuration files are available in the same project layout as the main HR-FRGS notebook.

In [19]:
import yaml
from pathlib import Path

# Adjust these paths as needed for your project structure
CONFIG_PATH = 'config/pipeline_config.yaml'
OUTPUT_DIR = Path('output_ablation')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

with open(CONFIG_PATH) as f:
    base_cfg = yaml.safe_load(f)

print('Loaded base config keys:', list(base_cfg.keys()))

Loaded base config keys: ['deseq2', 'ppi_filtering', 'wgcna', 'pruning', 'diffusion', 'scoring']


## Variant A: No Entropy Pruning (Phase 2)

This variant disables the entropy-based cluster filtering in Phase 2 while keeping other quality criteria unchanged.

Implementation strategy:
- Create a shallow copy of the config.
- Set a very high `entropy_threshold` so that no cluster is rejected due to entropy.
- Run Phase 2 with this modified config and save the resulting edges and metadata in a separate directory.


In [20]:
from copy import deepcopy
import pandas as pd

# Local helper to write a modified config for this ablation
ablation_cfg_A = deepcopy(base_cfg)
ablation_cfg_A.setdefault('ppi_filtering', {})['entropy_threshold'] = 1.0
# # Disabling density and weight thresholds by setting them to 0.0
# ablation_cfg_A['ppi_filtering']['density_threshold'] = 0.0
# ablation_cfg_A['ppi_filtering']['average_weight_threshold'] = 0.0

ABLATION_A_DIR = OUTPUT_DIR / 'no_entropy_pruning'
ABLATION_A_DIR.mkdir(exist_ok=True, parents=True)

# Save modified config
CFG_A_PATH = ABLATION_A_DIR / 'pipeline_config_no_entropy.yaml'
with open(CFG_A_PATH, 'w') as f:
    yaml.safe_dump(ablation_cfg_A, f)

print('Written ablation config for Variant A to', CFG_A_PATH)

Written ablation config for Variant A to output_ablation\no_entropy_pruning\pipeline_config_no_entropy.yaml


In [21]:
# Example driver for Phase 2 under Variant A
# Assumes you have a module exposing PPIEntropyFilter as in the main notebook

from phase2_ppi_entropy import PPIEntropyFilter  # adjust import if needed

filter_A = PPIEntropyFilter(config_path=str(CFG_A_PATH))
filtered_ppi_A, cluster_info_A = filter_A.run(
    degs_file='output/phase1_degs.csv',
    ppi_file='data/ppi_network.csv',
    expr_file='data/expression_matrix.csv',
    output_dir=str(ABLATION_A_DIR)
)

print('Variant A: filtered PPI edges:', len(filtered_ppi_A))
print('Variant A: clusters passing filter:', sum(1 for c in cluster_info_A if c.get('passes_filter', False)))

2026-04-29 21:30:45,846 - INFO - Starting PPI Entropy Filtering...
2026-04-29 21:30:45,850 - INFO - Loading input data...
2026-04-29 21:30:52,742 - INFO - DEGs to process: 5900
2026-04-29 21:30:52,745 - INFO - Total PPI interactions: 6499135
2026-04-29 21:30:53,877 - INFO - PPI interactions after DEG filter: 453382
2026-04-29 21:30:53,894 - INFO - PPI interactions after confidence score filter: 15399
2026-04-29 21:30:54,486 - INFO - Network: 2982 nodes, 15399 edges


Stage 1: Vertex Weighting...
Progress: 5.0% (Vertex Weighting)
Progress: 10.0% (Vertex Weighting)
Progress: 15.0% (Vertex Weighting)
Progress: 20.0% (Vertex Weighting)
Progress: 25.0% (Vertex Weighting)
Progress: 30.0% (Vertex Weighting)
Progress: 35.0% (Vertex Weighting)
Progress: 40.0% (Vertex Weighting)
Progress: 45.0% (Vertex Weighting)
Progress: 50.0% (Vertex Weighting)
Progress: 55.0% (Vertex Weighting)
Progress: 60.0% (Vertex Weighting)
Progress: 65.0% (Vertex Weighting)
Progress: 70.0% (Vertex Weighting)
Progress: 74.9% (Vertex Weighting)
Progress: 79.9% (Vertex Weighting)
Progress: 84.9% (Vertex Weighting)
Progress: 89.9% (Vertex Weighting)
Progress: 94.9% (Vertex Weighting)
Progress: 99.9% (Vertex Weighting)
Progress: 100.0% (Vertex Weighting)
Stage 2: Complex Detection / Expansion...
Progress: 5.0% (Complex Detection)
Progress: 15.0% (Complex Detection)
Progress: 20.0% (Complex Detection)
Progress: 25.0% (Complex Detection)
Progress: 30.0% (Complex Detection)
Progress: 40.0%

2026-04-29 21:31:37,876 - INFO - Clusters detected: 377


Progress: 68.7% (Post-processing)
Progress: 78.5% (Post-processing)
Progress: 88.3% (Post-processing)
Progress: 98.1% (Post-processing)
Progress: 100.0% (Post-processing)
MCODE completed.


2026-04-29 21:31:39,055 - INFO - Processed 50 clusters...
2026-04-29 21:31:39,305 - INFO - Processed 100 clusters...
2026-04-29 21:31:39,453 - INFO - Processed 150 clusters...
2026-04-29 21:31:39,519 - INFO - Processed 200 clusters...
2026-04-29 21:31:39,589 - INFO - Processed 250 clusters...
2026-04-29 21:31:39,623 - INFO - Processed 300 clusters...
2026-04-29 21:31:39,628 - INFO - Processed 350 clusters...
2026-04-29 21:31:39,642 - INFO - Clusters passing filter: 85 / 377
2026-04-29 21:31:39,643 - INFO - High-quality clusters after entropy filtering: 85
2026-04-29 21:31:41,643 - INFO - Filtered PPI saved: output_ablation\no_entropy_pruning\phase2_filtered_ppi.csv
2026-04-29 21:31:41,660 - INFO - Cluster info saved: output_ablation\no_entropy_pruning\phase2_clusters_info.json
2026-04-29 21:31:41,662 - INFO - 
Summary Statistics:
2026-04-29 21:31:41,663 - INFO -   total_edges_filtered: 3989
2026-04-29 21:31:41,664 - INFO -   total_unique_nodes: 609
2026-04-29 21:31:41,664 - INFO -   to

Variant A: filtered PPI edges: 3989
Variant A: clusters passing filter: 85


In [ ]:
# import subprocess

# # Define paths for Variant A
# ABLATION_A_DIR = OUTPUT_DIR / 'no_entropy_pruning'
# PHASE3_R_SCRIPT = 'Phase3.R'

# # Execute the R script
# # This assumes your R script accepts command line arguments for input/output
# print("Running Phase 3 (R) for Variant A...")
# subprocess.run([
#     'Rscript', PHASE3_R_SCRIPT, 
#     '--input_dir', str(ABLATION_A_DIR), 
#     '--output_dir', str(ABLATION_A_DIR)
# ], check=True)

In [22]:
from phase4 import HypergraphConstructor
from pathlib import Path

# 1. Setup Variant A Directory (Same as Phase 2/3)
ABLATION_A_DIR = OUTPUT_DIR / 'no_entropy_pruning'

# 2. Initialize Constructor
# Use your existing config; Phase 4 will read 'min_hyperedge_size' from it
constructor_A = HypergraphConstructor(config_path='config/pipeline_config.yaml')

# 3. Execute Phase 4 for Variant A
# We point to the 'unfiltered' files generated in ABLATION_A_DIR
H_A, metadata_A = constructor_A.run(
    struct_edges_file=str(ABLATION_A_DIR / 'phase2_filtered_ppi.csv'), # Unfiltered PPI
    func_adj_file=str(ABLATION_A_DIR / 'phase3_pruned_adjacency.csv'), # From ablated Phase 3
    degs_file='output/phase1_degs.csv',                                 # Standard DEGs
    metadata_file=str(ABLATION_A_DIR / 'phase2_clusters_info.json'),  # Unfiltered metadata
    phase3_info_file=str(ABLATION_A_DIR / 'phase3_power_info.txt'),    # From ablated Phase 3
    output_dir=str(ABLATION_A_DIR)
)

print(f"Variant A Hypergraph Construction Complete.")
print(f"Incidence Matrix Shape: {H_A.shape}") # Nodes x (Structural + Functional Edges)

2026-04-29 21:36:35,407 - INFO - Starting Hypergraph Construction...
2026-04-29 21:36:35,492 - INFO - Loading network data...
2026-04-29 21:36:43,421 - INFO - Total DEG nodes: 5900
2026-04-29 21:36:43,480 - INFO - Extracting structural hyperedges from PPI network...
2026-04-29 21:36:43,486 - INFO - Found 85 protein complexes
2026-04-29 21:36:43,487 - INFO - Structural hyperedges (PPI clusters): 85
2026-04-29 21:36:43,489 - INFO - Extracting functional hyperedges from WGCNA network...
2026-04-29 21:36:45,979 - INFO - Found 6 co-expression modules
2026-04-29 21:36:46,080 - INFO - Functional hyperedges (co-expression modules): 6
2026-04-29 21:36:46,083 - INFO - Total hyperedges: 91
2026-04-29 21:36:46,084 - INFO - Building incidence matrix...
2026-04-29 21:36:46,141 - INFO - Incidence matrix shape: (5900, 91)
2026-04-29 21:36:46,144 - INFO - Sparsity: 99.85%
2026-04-29 21:36:46,204 - INFO - Incidence matrix saved: output_ablation\no_entropy_pruning\phase4_hypergraph_incidence.npz
2026-04-

Variant A Hypergraph Construction Complete.
Incidence Matrix Shape: (5900, 91)


In [23]:
from phase5 import HypergraphDiffusionAnalyzer
from pathlib import Path

# 1. Setup Variant A Directory
ABLATION_A_DIR = OUTPUT_DIR / 'no_entropy_pruning'

# 2. Initialize the Analyzer for Variant A
# We use the standard config, as the 'ablation' here is in the input data itself
analyzer_A = HypergraphDiffusionAnalyzer(config_path='config/pipeline_config.yaml')

# 3. Execute Phase 5 for Variant A
# This uses the unfiltered incidence matrix and metadata from ABLATION_A_DIR
biomarkers_A_df = analyzer_A.run(
    hypergraph_file=str(ABLATION_A_DIR / 'phase4_hypergraph_incidence.npz'),
    metadata_file=str(ABLATION_A_DIR / 'phase4_hypergraph_metadata.json'),
    node_map_file=str(ABLATION_A_DIR / 'phase4_node_hyperedge_map.csv'),
    degs_file='output/phase1_degs.csv',
    probmap_file='data/gencode.v36.annotation.gtf.gene.probemap',
    output_dir=str(ABLATION_A_DIR)
)

print(f"Variant A Phase 5 Complete.")
print(f"Generated {len(biomarkers_A_df)} ranked biomarkers based on unfiltered hypergraph.")

2026-04-29 21:36:46,636 - INFO - Starting Hypergraph Diffusion Analysis...
2026-04-29 21:36:46,643 - INFO - Loading hypergraph data...
2026-04-29 21:36:47,220 - INFO - Slicing H to match specific gene indices...
2026-04-29 21:36:47,620 - INFO - Hypergraph: 5900 nodes, 91 hyperedges
2026-04-29 21:36:47,621 - INFO - Number of genes from node_map: 812
2026-04-29 21:36:47,621 - INFO - Number of genes from degs (full): 28670
2026-04-29 21:36:47,624 - INFO - Number of genes used for H: 812
2026-04-29 21:36:47,626 - INFO - Building transition matrix...
2026-04-29 21:36:47,662 - INFO - Running Random Walk with Restart...
2026-04-29 21:36:47,665 - INFO - RWR parameters: c=0.1, max_iter=1000, threshold=1e-06
2026-04-29 21:36:47,673 - INFO - RWR converged at iteration 1, diff=2.44e-11
2026-04-29 21:36:47,675 - INFO - Computing Hypergraph Betweenness Centrality...
2026-04-29 21:37:01,164 - INFO - Computing composite biomarker scores...
2026-04-29 21:37:01,278 - INFO - Final biomarkers saved: outpu

Variant A Phase 5 Complete.
Generated 622 ranked biomarkers based on unfiltered hypergraph.


In [24]:
from phase6 import HypergraphFeatureSelector
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Setup Variant A Directory
ABLATION_A_DIR = OUTPUT_DIR / 'no_entropy_pruning'

# 2. Data Alignment for Phase 6
# Load expression and metadata as used in previous phases
expr_df = pd.read_csv('data/expression_matrix.csv', index_col=0)
expr_df = np.log2(expr_df + 1)
meta_df = pd.read_csv('data/metadata.csv')

labels = (meta_df['Status'] == "Tumor").astype(int).values

# Use the ranked biomarkers from Variant A's Phase 5
genes_A = biomarkers_A_df['Gene'].values
bps_A = biomarkers_A_df['BiomarkerPotentialScore'].values

# Ensure expression matrix matches the ablated gene list
expr_df_aligned_A = expr_df.reindex(index=genes_A).dropna()

# 3. Initialize Selector with Original Logic
selector_A = HypergraphFeatureSelector(expr_df_aligned_A.values.T, bps_A, labels)

# 4. Perform Overlapping FCM Grouping (Original logic: threshold=0.20)
# This allows genes to belong to multiple clusters if they share functional signals
groups_A, reps_A = selector_A.fcm_grouping(n_clusters=max(1, len(genes_A)//50), threshold=0.20)

# 5. Autonomous Greedy Search
# Identify best candidates and find the optimal subset S_final_A
best_cands_A = [max(g['genes'], key=lambda x: selector_A.compute_ipd([x])) for g in groups_A]

print("Starting Autonomous Greedy Search for Variant A (Unfiltered, Overlapping)...")
S_final_A, hist_A, stab_hist_A, stab_sizes_A, k_opt_A = selector_A.greedy_search(np.array(best_cands_A))

# 6. Extract and Save Ablated Biomarkers
combined_pool_A = np.unique([g for grp in groups_A if any(s in grp['genes'] for s in S_final_A) for g in grp['genes']])
extracted_A = biomarkers_A_df.iloc[combined_pool_A].copy()

# Map cluster IDs and selection status
cluster_map_A = {g: grp['cluster_id'] for grp in groups_A for g in grp['genes']}
extracted_A['cluster_id'] = extracted_A.index.map(cluster_map_A)
extracted_A['is_selected'] = extracted_A.index.isin(S_final_A)

# Save result to ablation folder
OUT_PATH_A = ABLATION_A_DIR / 'EXTRACTED_BIOMARKERS_VARIANT_A.csv'
extracted_A.to_csv(OUT_PATH_A, index=False)

print(f'Variant A Complete. Saved {len(extracted_A)} biomarkers to {OUT_PATH_A}')

2026-04-29 21:37:02,869 - INFO - Initialized Selector. Sum(W)=1.0000, Max(W)=0.0036
2026-04-29 21:37:03,298 - INFO - Grouped into 12 overlapping clusters.
2026-04-29 21:37:03,299 - INFO - Total gene assignments: 933 (Total Genes: 622)


Starting Autonomous Greedy Search for Variant A (Unfiltered, Overlapping)...
Variant A Complete. Saved 181 biomarkers to output_ablation\no_entropy_pruning\EXTRACTED_BIOMARKERS_VARIANT_A.csv


## Variant B: No Fuzzy-Rough Set Based Selection (Phase 6)

This variant removes the fuzzy-rough feature selector and instead:
- Uses the **BiomarkerPotentialScore (BPS)** from Phase 5 directly.
- Selects the top-\(k\) genes by BPS as the final biomarker set.

You can tune `TOP_K` or derive it from the size of the original selected set.


In [25]:
import numpy as np

TOP_K = 100  # adjust as desired or match original selector size

BIOMARKERS_PATH = Path('output/BIOMARKERS_FINAL.csv')

bio_df = pd.read_csv(BIOMARKERS_PATH)

bio_df = bio_df.sort_values('BiomarkerPotentialScore', ascending=False)
selected_B = bio_df.head(TOP_K).copy()

B_DIR = OUTPUT_DIR / 'no_fuzzy_rough'
B_DIR.mkdir(exist_ok=True, parents=True)

OUT_B_PATH = B_DIR / 'BIOMARKERS_NO_FRGS_TOPK.csv'
selected_B.to_csv(OUT_B_PATH, index=False)

print('Variant B: saved top-{} biomarkers (no fuzzy-rough selection) to'.format(TOP_K), OUT_B_PATH)

Variant B: saved top-100 biomarkers (no fuzzy-rough selection) to output_ablation\no_fuzzy_rough\BIOMARKERS_NO_FRGS_TOPK.csv


### Optional: Evaluation Hook for Variant B

Plug in your downstream classifier or cross-validation protocol here to compare performance with the full HR-FRGS pipeline.


In [ ]:
# Skeleton: user-defined evaluation
# from my_evaluation_module import evaluate_biomarkers
# score_B = evaluate_biomarkers(selected_B['Gene'].tolist())
# print('Variant B evaluation score:', score_B)